In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


# ───────────────────────────────────────────────────────────────────────────
# SECTION 1 — SYNTHETIC DATA GENERATION
# Realistic design:
#   Normal   : tight CV (10-15%) — stays within peer price band
#   Anomaly  : 2x-4x over, 20%-55% under — realistic TBML range (FATF 2020)
#   5% anomaly prevalence — calibrated to Paper Table 1
# ───────────────────────────────────────────────────────────────────────────

N_TRANSACTIONS = 10_000
ANOMALY_RATE = 0.05
OVER_RATIO = 0.60

# (lo, hi, CV) — normal prices have CV=10-15% of midpoint
PRODUCT_CATEGORIES = {
    "electronics": (300, 600, 0.10),
    "textile": (8, 20, 0.12),
    "machinery": (800, 2000, 0.10),
    "chemicals": (60, 180, 0.12),
    "food_products": (2, 8, 0.15),
    "furniture": (100, 400, 0.12),
    "raw_materials": (20, 60, 0.10),
    "medical_devices": (400, 1200, 0.10),
}

HIGH_RISK_COUNTRIES = ["Panama", "UAE", "Cayman Islands", "Malta", "Seychelles"]
ALL_COUNTRIES = [
    "Germany",
    "Japan",
    "UK",
    "Singapore",
    "Canada",
    "Australia",
    "China",
    "India",
    "Vietnam",
    "Turkey",
] + HIGH_RISK_COUNTRIES

rows = []
for i in range(N_TRANSACTIONS):
    cat = rng.choice(list(PRODUCT_CATEGORIES))
    lo, hi, cv = PRODUCT_CATEGORIES[cat]
    mid = (lo + hi) / 2
    std = mid * cv  # tight std — realistic peer variance
    qty = max(1, min(int(rng.lognormal(4.0, 0.8)), 5000))
    origin = rng.choice(ALL_COUNTRIES)
    hr = 1 if origin in HIGH_RISK_COUNTRIES else 0
    imp_id = rng.integers(1, 300)
    is_anom = 1 if rng.random() < ANOMALY_RATE else 0

    if is_anom:
        # 40% obvious extreme cases, 60% subtle borderline cases
        is_extreme = rng.random() < 0.40

        if rng.random() < OVER_RATIO:
            direction = "OVER_INVOICING"
            if is_extreme:
                mult = rng.uniform(2.0, 4.0)  # clear-cut (FATF documented)
            else:
                mult = rng.uniform(1.25, 1.8)  # subtle — needs multivariate signal
            unit_price = round(rng.uniform(hi * mult * 0.92, hi * mult * 1.08), 2)
        else:
            direction = "UNDER_INVOICING"
            if is_extreme:
                mult = rng.uniform(0.20, 0.55)  # clear-cut
            else:
                mult = rng.uniform(0.55, 0.80)  # subtle
            unit_price = round(rng.uniform(lo * mult * 0.92, lo * mult * 1.08), 2)
        unit_price = max(0.01, unit_price)
    else:
        direction = "NORMAL"
        # Tight normal — realistic peer price band
        unit_price = max(0.01, round(rng.normal(mid, std), 2))
        unit_price = max(0.01, round(rng.normal(mid, std), 2))

    rows.append(
        {
            "txn_id": f"TXN{i+1:05d}",
            "product_group": cat,
            "quantity": qty,
            "unit_price": unit_price,
            "declared_value": round(unit_price * qty, 2),
            "country_origin": origin,
            "country_risk": hr,
            "importer_id": imp_id,
            "anomaly_label": is_anom,
            "true_direction": direction,
        }
    )

df = pd.DataFrame(rows)
y = df["anomaly_label"].values

total = len(df)
n_anom = int(y.sum())
n_normal = total - n_anom

print("=" * 60)
print("  TBML Detection — Hybrid Multiplicative Ensemble")
print("=" * 60)
print(f"  Total          : {total:,}")
print(f"  Normal         : {n_normal:,} ({n_normal/total*100:.1f}%)")
print(f"  Anomalies      : {n_anom:,}  ({n_anom/total*100:.1f}%)")
print(f"  Over-invoicing : {(df.true_direction=='OVER_INVOICING').sum()}")
print(f"  Under-invoicing: {(df.true_direction=='UNDER_INVOICING').sum()}")

from sklearn.model_selection import train_test_split

# ───────────────────────────────────────────────────────────────────────────
# SECTION 1B — TRAIN/TEST SPLIT (prevents circular evaluation)
# ───────────────────────────────────────────────────────────────────────────
TEST_SIZE = 0.20

train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    stratify=y,  # anomaly ratio same in both splits
    random_state=RANDOM_SEED,
)

df["split"] = "test"
df.loc[train_idx, "split"] = "train"

print(f"Train size : {len(train_idx)}  (anomalies: {y[train_idx].sum()})")
print(f"Test size  : {len(test_idx)}   (anomalies: {y[test_idx].sum()})")

train_mask = (df["split"] == "train").values
test_mask = (df["split"] == "test").values

# ───────────────────────────────────────────────────────────────────────────
# SECTION 2 — FEATURE ENGINEERING (statistics learned from TRAIN ONLY)
# ───────────────────────────────────────────────────────────────────────────

train_df = df[df["split"] == "train"]

# Peer median & MAD — computed from train, mapped to everyone
peer_median_map = train_df.groupby("product_group")["unit_price"].median()
peer_mad_map = train_df.groupby("product_group")["unit_price"].apply(
    lambda x: (x - x.median()).abs().median()
)
df["peer_median"] = df["product_group"].map(peer_median_map)
df["MAD"] = df["product_group"].map(peer_mad_map)

df["robust_z"] = (df["unit_price"] - df["peer_median"]) / (df["MAD"] + 1e-9)
df["abs_robust_z"] = df["robust_z"].abs()

df["price_ratio"] = df["unit_price"] / (df["peer_median"] + 1e-9)
df["log_price_ratio"] = np.log1p(np.abs(df["price_ratio"] - 1))

# Value anomaly — mean/std from train only
value_mean = train_df["declared_value"].mean()
value_std = train_df["declared_value"].std()
df["value_zscore"] = (df["declared_value"] - value_mean) / value_std

# Quantity anomaly — mean/std from train only
df["log_qty"] = np.log1p(df["quantity"])
qty_mean = np.log1p(train_df["quantity"]).mean()
qty_std = np.log1p(train_df["quantity"]).std()
df["qty_zscore"] = (df["log_qty"] - qty_mean) / qty_std

# Importer frequency — counted from train only
freq_map = train_df["importer_id"].value_counts()
df["importer_freq"] = df["importer_id"].map(freq_map).fillna(0)

FEATURES = [
    "unit_price",
    "peer_median",
    "robust_z",
    "abs_robust_z",
    "price_ratio",
    "log_price_ratio",
    "value_zscore",
    "qty_zscore",
    "importer_freq",
    "country_risk",
]

# Scaler fit on TRAIN only, then applied to everyone
scaler = StandardScaler()
X_scaled = np.zeros((len(df), len(FEATURES)))
X_scaled[train_mask] = scaler.fit_transform(df.loc[train_mask, FEATURES].fillna(0))
X_scaled[test_mask] = scaler.transform(df.loc[test_mask, FEATURES].fillna(0))

print(f"\n  Features       : {len(FEATURES)}")


# ───────────────────────────────────────────────────────────────────────────
# SECTION 3 — DETECTORS
# Contamination is NOT fixed at the true anomaly rate — searched over a
# plausible range [0.005, 0.05] since true rate is unknown in practice.
# Selected via train-only Average Precision (threshold-free, leak-free).
# ───────────────────────────────────────────────────────────────────────────


def normalize_01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)


CONTAMINATION_GRID = np.arange(0.005, 0.055, 0.005)

best_ap_iso, best_contam, best_iso_forest = -1.0, None, None

for c_val in CONTAMINATION_GRID:
    candidate_forest = IsolationForest(
        n_estimators=300,
        contamination=c_val,
        max_samples="auto",
        random_state=42,
        n_jobs=-1,
    )
    candidate_forest.fit(X_scaled[train_mask])
    candidate_score = normalize_01(-candidate_forest.decision_function(X_scaled))

    ap_train = average_precision_score(y[train_mask], candidate_score[train_mask])

    if ap_train > best_ap_iso:
        best_ap_iso = ap_train
        best_contam = c_val
        best_iso_forest = candidate_forest
        iso_score = candidate_score

print(f"  Selected contamination (train AP={best_ap_iso:.4f}) : {best_contam:.3f}")

# Statistical Robust Z-score
stat_score = normalize_01(df["abs_robust_z"].values)

# Price ratio signal — domain amplifier
ratio_signal = normalize_01(df["log_price_ratio"].values)


# ───────────────────────────────────────────────────────────────────────────
# SECTION 4 — MULTIPLICATIVE ENSEMBLE (weights learned from TRAIN ONLY)
#
# log(score) = a·log(stat) + b·log(iso) + c·log(ratio),  a+b+c=1, a,b,c≥0
# Weights selected to maximize Average Precision (threshold-free) on train.
# ───────────────────────────────────────────────────────────────────────────

WEIGHT_STEP = 0.05
best_ap, best_weights = -1.0, None

for a in np.arange(0.0, 1.0 + WEIGHT_STEP, WEIGHT_STEP):
    for b in np.arange(0.0, 1.0 - a + WEIGHT_STEP, WEIGHT_STEP):
        c = round(1.0 - a - b, 4)
        if c < 0:
            continue

        candidate_score = (stat_score**a) * (iso_score**b) * (ratio_signal**c)
        candidate_score = normalize_01(candidate_score)

        ap_train = average_precision_score(y[train_mask], candidate_score[train_mask])

        if ap_train > best_ap:
            best_ap = ap_train
            best_weights = (round(a, 2), round(b, 2), c)
            ensemble_score = candidate_score

a_opt, b_opt, c_opt = best_weights
print(f"\n  Learned weights (train AP={best_ap:.4f}):")
print(f"    stat exponent  (a) = {a_opt}")
print(f"    iso exponent   (b) = {b_opt}")
print(f"    ratio exponent (c) = {c_opt}")

best_f1, best_thr = 0.0, 0.0
for pct in np.arange(93.0, 99.5, 0.1):
    thr = np.percentile(ensemble_score[train_mask], pct)  # train-only percentile
    pred_train = (ensemble_score[train_mask] >= thr).astype(int)
    f1 = f1_score(y[train_mask], pred_train, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

# Fixed threshold now applied to EVERYONE (train + test)
pred = (ensemble_score >= best_thr).astype(int)

# Direction detection
Z_THRESHOLD = 2.5


def get_direction(z):
    if z > Z_THRESHOLD:
        return "OVER_INVOICING"
    elif z < -Z_THRESHOLD:
        return "UNDER_INVOICING"
    else:
        return "NORMAL"


df["ensemble_score"] = ensemble_score
df["ensemble_pred"] = pred
df["risk_level"] = pd.Series(pred).map({1: "HIGH", 0: "LOW"}).values
df["predicted_direction"] = df["robust_z"].apply(get_direction)


# Business explanation
def explain(row):
    parts = []
    z = row["robust_z"]
    if abs(z) > Z_THRESHOLD:
        side = "above" if z > 0 else "below"
        parts.append(
            f"Unit price {abs(z):.1f}σ {side} peer median "
            f"→ {row['predicted_direction'].replace('_',' ').lower()}"
        )
    if row["ensemble_pred"] == 1 and abs(z) <= Z_THRESHOLD:
        parts.append("Multivariate pattern anomalous (Isolation Forest)")
    if row["country_risk"] == 1:
        parts.append("High-risk trade corridor")
    return " | ".join(parts) if parts else "No significant risk signals"


df["explanation"] = df.apply(explain, axis=1)


# ───────────────────────────────────────────────────────────────────────────
# SECTION 5 — EVALUATION
# ───────────────────────────────────────────────────────────────────────────

p = precision_score(y[test_mask], pred[test_mask], zero_division=0)
r = recall_score(y[test_mask], pred[test_mask], zero_division=0)
f1 = f1_score(y[test_mask], pred[test_mask], zero_division=0)
f2 = fbeta_score(y[test_mask], pred[test_mask], beta=2, zero_division=0)
auc = roc_auc_score(y[test_mask], ensemble_score[test_mask])
ap = average_precision_score(y[test_mask], ensemble_score[test_mask])
ba = balanced_accuracy_score(y[test_mask], pred[test_mask])
cm = confusion_matrix(y[test_mask], pred[test_mask])
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)
fn_r = fn / y[test_mask].sum()

print("\n" + "=" * 60)
print("  EVALUATION RESULTS")
print("=" * 60)
print(f"  TP = {tp:4d}  (real fraud caught)")
print(f"  FP = {fp:4d}  (false alarms)")
print(f"  TN = {tn:4d}  (normal correctly passed)")
print(f"  FN = {fn:4d}  (fraud missed)")
print("-" * 60)

METRICS = [
    ("Precision", p, "≥ 95%", p >= 0.95),
    ("Recall", r, "≥ 95%", r >= 0.95),
    ("F1-Score", f1, "≥ 95%", f1 >= 0.95),
    ("F2-Score", f2, "≥ 95%", f2 >= 0.95),
    ("AUC-ROC", auc, "≥ 95%", auc >= 0.95),
    ("Avg Precision", ap, "≥ 95%", ap >= 0.95),
    ("Balanced Acc.", ba, "≥ 95%", ba >= 0.95),
    ("FPR", fpr, "~ 1%", fpr <= 0.01),
    ("FN rate", fn_r, "~ 1%", fn_r <= 0.01),
]

for name, val, target, passed in METRICS:
    tick = "✓" if passed else "✗"
    print(f"  {tick} {name:<20} {val*100:>7.2f}%   target {target}")

print("=" * 60)
print("\n  Classification Report:")
print(
    classification_report(
        y[test_mask], pred[test_mask], target_names=["Normal", "Suspicious"], digits=3
    )
)

print("=" * 60)
print("  TOP 10 HIGH-RISK TRANSACTIONS")
print("=" * 60)
top = (
    df[df["risk_level"] == "HIGH"]
    .sort_values("ensemble_score", ascending=False)
    .head(10)[
        [
            "txn_id",
            "product_group",
            "unit_price",
            "peer_median",
            "robust_z",
            "predicted_direction",
        ]
    ]
)
pd.set_option("display.width", 200)
print(top.to_string(index=False))

print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
print(f"  Flagged HIGH       : {int(pred.sum())}")
print(f"  Over-invoicing     : {(df.predicted_direction=='OVER_INVOICING').sum()}")
print(f"  Under-invoicing    : {(df.predicted_direction=='UNDER_INVOICING').sum()}")
print(f"  FPR                : {fpr*100:.2f}%")
print(f"  FN rate            : {fn_r*100:.2f}%")
print(f"  Ensemble threshold : {best_thr:.4f}")
print(f"  Ensemble formula   : stat^{a_opt} × IF^{b_opt} × ratio^{c_opt}")
print("=" * 60)

# Save
out_cols = [
    "txn_id",
    "product_group",
    "quantity",
    "unit_price",
    "declared_value",
    "country_origin",
    "country_risk",
    "peer_median",
    "robust_z",
    "abs_robust_z",
    "ensemble_score",
    "risk_level",
    "predicted_direction",
    "anomaly_label",
    "true_direction",
    "explanation",
]
df[out_cols].to_csv("tbml_results_final.csv", index=False)
print("\n  Saved → tbml_results_final.csv")

  TBML Detection — Hybrid Multiplicative Ensemble
  Total          : 10,000
  Normal         : 9,505 (95.0%)
  Anomalies      : 495  (5.0%)
  Over-invoicing : 292
  Under-invoicing: 203
Train size : 8000  (anomalies: 396)
Test size  : 2000   (anomalies: 99)

  Features       : 10
  Selected contamination (train AP=0.8849) : 0.005

  Learned weights (train AP=0.9999):
    stat exponent  (a) = 0.45
    iso exponent   (b) = 0.3
    ratio exponent (c) = 0.25

  EVALUATION RESULTS
  TP =   99  (real fraud caught)
  FP =    1  (false alarms)
  TN = 1900  (normal correctly passed)
  FN =    0  (fraud missed)
------------------------------------------------------------
  ✓ Precision              99.00%   target ≥ 95%
  ✓ Recall                100.00%   target ≥ 95%
  ✓ F1-Score               99.50%   target ≥ 95%
  ✓ F2-Score               99.80%   target ≥ 95%
  ✓ AUC-ROC               100.00%   target ≥ 95%
  ✓ Avg Precision         100.00%   target ≥ 95%
  ✓ Balanced Acc.          99.97%   

In [2]:
import pandas as pd

tbml = pd.read_csv("tbml_results_final.csv")
tbml

,txn_id,product_group,quantity,unit_price,declared_value,country_origin,country_risk,peer_median,robust_z,abs_robust_z,ensemble_score,risk_level,predicted_direction,anomaly_label,true_direction,explanation
0,TXN00001,electronics,23,391.40,9002.20,UAE,1,448.440,-1.889367,1.889367,0.059133,LOW,NORMAL,0,NORMAL,High-risk trade corridor
1,TXN00002,raw_materials,60,43.52,2611.20,Panama,1,40.250,1.056543,1.056543,0.039234,LOW,NORMAL,0,NORMAL,High-risk trade corridor
2,TXN00003,textile,57,14.62,833.34,Malta,1,14.050,0.483051,0.483051,0.019033,LOW,NORMAL,0,NORMAL,High-risk trade corridor
3,TXN00004,raw_materials,25,37.28,932.00,Cayman Islands,1,40.250,-0.959612,0.959612,0.030351,LOW,NORMAL,0,NORMAL,High-risk trade corridor
4,TXN00005,electronics,48,466.44,22389.12,Seychelles,1,448.440,0.596224,0.596224,0.025259,LOW,NORMAL,0,NORMAL,High-risk trade corridor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TXN09996,raw_materials,101,43.59,4402.59,Singapore,0,40.250,1.079160,1.079160,0.023711,LOW,NORMAL,0,NORMAL,No significant risk signals
9996,TXN09997,textile,74,14.98,1108.52,Canada,0,14.050,0.788136,0.788136,0.012999,LOW,NORMAL,0,NORMAL,No significant risk signals
9997,TXN09998,machinery,105,1349.29,141675.45,Canada,0,1411.000,-0.592738,0.592738,0.033273,LOW,NORMAL,0,NORMAL,No significant risk signals
9998,TXN09999,chemicals,51,113.52,5789.52,Cayman Islands,1,119.840,-0.623889,0.623889,0.022260,LOW,NORMAL,0,NORMAL,High-risk trade corridor
